In [ ]:
import rasterio
import matplotlib.pyplot as plt

In [ ]:
tif_path = "/data/ICESat-2/ATL18-gl_te_mean_all_1000m_20180101000000-20250701000000_003_01 1.tif"

with rasterio.open(tif_path) as src:
    print(f"CRS: {src.crs}")
    print(f"Shape: {src.shape}")
    print(f"Bounds: {src.bounds}")
    print(f"Bands: {src.count}")

In [ ]:
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds
import numpy as np

dst_crs = "EPSG:4326"
dst_res = 0.25  # degrees

# Full global extent to match Three.js equirectangular sphere UV mapping
width = int(360 / dst_res)   # 1440
height = int(180 / dst_res)  # 720
transform = from_bounds(-180, -90, 180, 90, width, height)

with rasterio.open(tif_path) as src:
    reprojected = np.empty((src.count, height, width), dtype=src.dtypes[0])

    for i in range(1, src.count + 1):
        reproject(
            source=rasterio.band(src, i),
            destination=reprojected[i - 1],
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
        )

In [ ]:
# Mask values greater than 10km (10000m)
reprojected = reprojected.astype(np.float64)
reprojected[reprojected > 10000] = np.nan

print(f"Reprojected shape: {reprojected.shape}")
print(f"Resolution: {dst_res}° x {dst_res}°")
print(f"Transform:\n{transform}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
data = reprojected[0]
data = np.where(data == 0, np.nan, data)  # mask nodata
im = ax.imshow(data, extent=[
    transform.c, transform.c + transform.a * width,
    transform.f + transform.e * height, transform.f
], aspect='auto')
plt.colorbar(im, ax=ax, label='Value')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('ATL18 - Reprojected to WGS84 (0.25°)')
plt.show()

In [ ]:
from PIL import Image
from matplotlib.colors import LogNorm
import matplotlib.cm as cm

data = reprojected[0].copy()
data[data == 0] = np.nan

# Log-scale normalization to accentuate lower values
vmin = max(np.nanmin(data), 1.0)  # clamp min to 1 for log scale
norm = LogNorm(vmin=vmin, vmax=np.nanmax(data))
mapped = cm.jet(norm(data))  # RGBA float array
mapped = (mapped * 255).astype(np.uint8)

# Make NaN pixels fully transparent
mask = np.isnan(data)
mapped[mask] = [0, 0, 0, 0]

img = Image.fromarray(mapped, mode='RGBA')
img.save('/data/web/ATL18_reprojected.png')
print(f'Saved PNG: {img.size[0]}x{img.size[1]} pixels')